# 🐟 Fish Speech 1.5 — Standalone LoRA Inference

**Upload these 3 files to `/content/` before running:**
```
/content/step_000000600.ckpt   ← trained LoRA checkpoint
/content/00001.wav             ← reference speaker audio
/content/00001.lab             ← transcript of reference audio
```
The `.npy` prompt token file is **auto-extracted** from your wav in Cell 5.

---
**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

## ⚙️ Cell 1 — Install Dependencies

In [1]:
!apt-get update -qq && apt-get install -y -qq ffmpeg build-essential cmake \
    libasound-dev portaudio19-dev libportaudio2 libportaudiocpp0
!pip install -q huggingface_hub librosa soundfile

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
dpkg: libjack-jackd2-0:amd64: dependency problems, but removing anyway as you requested:
 libavdevice58:amd64 depends on libjack-jackd2-0 (>= 1.9.10+20150825) | libjack-0.125; however:
  Package libjack-jackd2-0:amd64 is to be removed.
  Package libjack-0.125 is not installed.
  Package libjack-jackd2-0:amd64 which provides libjack-0.125 is to be removed.
 libavdevice58:amd64 depends on libjack-jackd2-0 (>= 1.9.10+20150825) | libjack-0.125; however:
  Package libjack-jackd2-0:amd64 is to be removed.
  Package libjack-0.125 is not installed.
  Package libjack-jackd2-0:amd64 which provides libjack-0.125 is to be removed.

(Reading database ... 118194 files and directories currently installed.)
Removing libjack-jackd2-0:amd64 (1.9.20~dfsg-1) ...
Selecting previously unselected package libjack0:amd64.
(R

## 📦 Cell 2 — Clone Repo & Install Fish Speech

In [2]:
import os
REPO_DIR = '/content/fish-speech'
%cd /content
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/fishaudio/fish-speech.git
else:
    print('✅ Repo already cloned')
%cd /content/fish-speech
!git fetch --tags -q
!git checkout v1.5.1 -q
!pip install -q -e .

/content
Cloning into 'fish-speech'...
remote: Enumerating objects: 6565, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 6565 (delta 191), reused 109 (delta 109), pack-reused 6311 (from 3)
Receiving objects: 100% (6565/6565), 28.32 MiB | 18.37 MiB/s, done.
Resolving deltas: 100% (4236/4236), done.
/content/fish-speech
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━

## ⬇️ Cell 3 — Download Base Model Weights (~1.5 GB)

In [6]:
# Cell 3 — Download Base Model (from your original script)
import os

BASE_CKPT = '/content/fish-speech/checkpoints/fish-speech-1.5'

if not os.path.exists(BASE_CKPT):
    print('⬇️  Downloading Fish Speech 1.5 base weights (~1.5 GB)...')
    !pip install -q huggingface_hub
    !python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='fishaudio/fish-speech-1.5', local_dir='{BASE_CKPT}')"
    print('✅ Done')
else:
    print('✅ Base model already present')

VQGAN_CKPT = f'{BASE_CKPT}/firefly-gan-vq-fsq-8x1024-21hz-generator.pth'
print(f'VQGAN exists: {os.path.exists(VQGAN_CKPT)}')

⬇️  Downloading Fish Speech 1.5 base weights (~1.5 GB)...

Fetching 7 files: 100% 7/7 [00:15<00:00,  2.25s/it]
Download complete: : 1.47GB [00:15, 92.7MB/s]
✅ Done
VQGAN exists: True


## 🔧 Cell 4 — Configuration & Path Check
> **Edit the top block if your filenames differ**

In [7]:
import os

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  EDIT HERE — all files live directly in /content/
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LORA_CKPT   = '/content/step_000000600.ckpt'
PROMPT_WAV  = '/content/00001.wav'
PROMPT_LAB  = '/content/00001.lab'
PROMPT_NPY  = '/content/00001.npy'               # ← now uploaded directly
TARGET_TEXT = 'Өнөөдөр гадаа хасах хорин хэм хүйтэн байна.'
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Derived — no need to change
REPO_DIR    = '/content/fish-speech'
BASE_CKPT   = f'{REPO_DIR}/checkpoints/fish-speech-1.5'
VQGAN_CKPT  = f'{BASE_CKPT}/firefly-gan-vq-fsq-8x1024-21hz-generator.pth'
MERGED_DIR  = f'{REPO_DIR}/checkpoints/fish-speech-1.5-merged'
LORA_CONFIG = 'r_8_alpha_16'
OUTPUT_WAV  = '/content/output.wav'
TEMP_DIR    = f'{REPO_DIR}/temp'

# Validate ALL files including npy
checks = {
    'LoRA checkpoint' : LORA_CKPT,
    'Prompt wav'      : PROMPT_WAV,
    'Prompt lab'      : PROMPT_LAB,
    'Prompt npy'      : PROMPT_NPY,
    'Base checkpoint' : BASE_CKPT,
    'VQGAN weights'   : VQGAN_CKPT,
}
all_ok = True
for label, path in checks.items():
    ok = os.path.exists(path)
    print(f"{'✅' if ok else '❌  MISSING'}  {label}: {path}")
    if not ok: all_ok = False

print()
if all_ok:
    with open(PROMPT_LAB, 'r', encoding='utf-8') as f:
        PROMPT_TEXT = f.read().strip()
    print(f'📜 Prompt text : {PROMPT_TEXT}')
    print(f'🎙️  Target text : {TARGET_TEXT}')
    print('\n✅ Ready — proceed to Cell 6 (skip Cell 5)')
else:
    print('\n❌ Upload the missing files to /content/ and re-run this cell.')

✅  LoRA checkpoint: /content/step_000000600.ckpt
✅  Prompt wav: /content/00001.wav
✅  Prompt lab: /content/00001.lab
✅  Prompt npy: /content/00001.npy
✅  Base checkpoint: /content/fish-speech/checkpoints/fish-speech-1.5
✅  VQGAN weights: /content/fish-speech/checkpoints/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth

📜 Prompt text : Монгол Улсын бизнесийн орчин сүүлийн жилүүдэд эрс өөрчлөгдөж байна.
🎙️  Target text : Өнөөдөр гадаа хасах хорин хэм хүйтэн байна.

✅ Ready — proceed to Cell 6 (skip Cell 5)


## 🔑 Cell 5 — Extract Prompt Tokens (.npy) from Wav
> Runs once per reference wav. Saves `/content/00001.npy`.

In [8]:
# import os
# %cd /content/fish-speech

# if os.path.exists(PROMPT_NPY):
#     print(f'✅ NPY already exists: {PROMPT_NPY}')
# else:
#     print(f'⚙️  Extracting VQ tokens from {PROMPT_WAV} ...')
#     # extract_vq processes a folder; point it at /content/ so it finds the wav
#     !python tools/vqgan/extract_vq.py /content \
#         --num-workers 1 \
#         --batch-size  4 \
#         --config-name 'firefly_gan_vq' \
#         --checkpoint-path '{VQGAN_CKPT}'

#     if os.path.exists(PROMPT_NPY):
#         print(f'\n✅ Saved → {PROMPT_NPY}')
#     else:
#         print('\n❌ Extraction failed — check output above.')

## 🔗 Cell 6 — Merge LoRA into Base Model

In [ ]:
import os
%cd /content/fish-speech

if os.path.isdir(MERGED_DIR):
    print(f'⏭  Already merged → {MERGED_DIR}')
    print('   Delete that folder to force re-merge.')
else:
    print(f'🔗 Merging LoRA weights...\n   LoRA : {LORA_CKPT}\n   Base : {BASE_CKPT}\n   → Out: {MERGED_DIR}\n')
    !python tools/llama/merge_lora.py \
        --lora-config '{LORA_CONFIG}' \
        --base-weight '{BASE_CKPT}' \
        --lora-weight '{LORA_CKPT}' \
        --output      '{MERGED_DIR}'
    if os.path.isdir(MERGED_DIR):
        print(f'\n✅ Merge complete → {MERGED_DIR}')
    else:
        print('\n❌ Merge failed — check the output above.')

## 🧠 Cell 7 — Text → Semantic Tokens

In [14]:
import glob, shutil, os, torch
%cd /content/fish-speech

if os.path.exists(TEMP_DIR): shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)
if os.path.exists(OUTPUT_WAV): os.remove(OUTPUT_WAV)

with open(PROMPT_LAB, 'r', encoding='utf-8') as f:
    PROMPT_TEXT = f.read().strip()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📜 Prompt : {PROMPT_TEXT}\n🎙️  Target : {TARGET_TEXT}\n🖥️  Device : {DEVICE}\n')

!python fish_speech/models/text2semantic/inference.py \
    --text            '{TARGET_TEXT}' \
    --prompt-text     '{PROMPT_TEXT}' \
    --prompt-tokens   '{PROMPT_NPY}' \
    --checkpoint-path '{MERGED_DIR}' \
    --device          {DEVICE}

npy_files = sorted(glob.glob(f'{TEMP_DIR}/codes_*.npy'))
if npy_files:
    print(f'\n✅ Tokens ready → {npy_files[0]}')
else:
    print('\n❌ No token files in temp/ — semantic generation failed.')

/content/fish-speech
📜 Prompt : Монгол Улсын бизнесийн орчин сүүлийн жилүүдэд эрс өөрчлөгдөж байна.
🎙️  Target : Өнөөдөр гадаа хасах хорин хэм хүйтэн байна.
🖥️  Device : cpu

2026-03-29 03:46:17.123 | INFO     | __main__:main:1056 - Loading model ...
2026-03-29 03:46:27.519 | INFO     | __main__:load_model:681 - Restored model from checkpoint
2026-03-29 03:46:27.520 | INFO     | __main__:load_model:687 - Using DualARTransformer
2026-03-29 03:46:27.591 | INFO     | __main__:main:1070 - Time to load model: 10.47 seconds
2026-03-29 03:46:27.638 | INFO     | __main__:generate_long:788 - Encoded text: Өнөөдөр гадаа хасах хорин хэм хүйтэн байна.
2026-03-29 03:46:27.638 | INFO     | __main__:generate_long:806 - Generating sentence 1/1 of sample 1/1
  1% 63/7976 [01:23<2:55:03,  1.33s/it]
2026-03-29 03:48:14.241 | INFO     | __main__:generate_long:860 - Generated 65 tokens in 106.60 seconds, 0.61 tokens/sec
2026-03-29 03:48:14.241 | INFO     | __main__:generate_long:863 - Bandwidth achieved: 0

## 🔊 Cell 8 — Decode Tokens → Audio

In [16]:
import glob, os, torch
%cd /content/fish-speech

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

npy_files = sorted(glob.glob(f'{TEMP_DIR}/codes_*.npy'))
if not npy_files:
    print('❌ No token files found. Run Cell 7 first.')
else:
    print(f'🎵 Decoding {npy_files[0]} → {OUTPUT_WAV}  [{DEVICE}]\n')
    !python fish_speech/models/vqgan/inference.py \
        -i '{npy_files[0]}' \
        --checkpoint-path '{VQGAN_CKPT}' \
        --output-path     '{OUTPUT_WAV}' \
        --device          {DEVICE}
    if os.path.exists(OUTPUT_WAV):
        print(f'\n✅ Saved → {OUTPUT_WAV}  ({os.path.getsize(OUTPUT_WAV)/1024:.1f} KB)')
    else:
        print('\n❌ Decoding failed.')

/content/fish-speech
🎵 Decoding /content/fish-speech/temp/codes_0.npy → /content/output.wav  [cpu]

/usr/local/lib/python3.12/dist-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:445: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:630: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/vector_quantize_pytorch/finite_scalar_quantization.py:147: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/vector_quantize_pytorch/lookup_free_quantization.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is dep

## 🎧 Cell 9 — Listen & Download

In [17]:
import IPython.display as ipd, os

print('🎤 Reference voice:')
ipd.display(ipd.Audio(PROMPT_WAV))

print('\n✨ Generated speech:')
if os.path.exists(OUTPUT_WAV):
    ipd.display(ipd.Audio(OUTPUT_WAV))
else:
    print('❌ output.wav not found — run Cells 7 & 8 first.')

# Download (Colab only)
try:
    from google.colab import files
    if os.path.exists(OUTPUT_WAV):
        print('\n⬇️  Downloading output.wav...')
        files.download(OUTPUT_WAV)
except ImportError:
    print(f'\nℹ️  File saved at: {OUTPUT_WAV}')

🎤 Reference voice:



✨ Generated speech:



⬇️  Downloading output.wav...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🔁 Cell 10 — Quick Re-run (new text, same voice)
> Skips merge. Only edit `NEW_TEXT`.

In [13]:
import glob, shutil, os
import IPython.display as ipd

# ── Edit only this ─────────────────────────────────────────
NEW_TEXT   = 'Сайн байна уу? Та хэрхэн байна?'
NEW_OUTPUT = '/content/output_new.wav'
# ──────────────────────────────────────────────────────────

%cd /content/fish-speech
if os.path.exists(TEMP_DIR): shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)

print(f'🎙️  Generating: {NEW_TEXT}\n')

!python fish_speech/models/text2semantic/inference.py \
    --text            '{NEW_TEXT}' \
    --prompt-text     '{PROMPT_TEXT}' \
    --prompt-tokens   '{PROMPT_NPY}' \
    --checkpoint-path '{MERGED_DIR}'

npy_files = sorted(glob.glob(f'{TEMP_DIR}/codes_*.npy'))
if not npy_files:
    print('❌ No tokens generated.')
else:
    !python fish_speech/models/vqgan/inference.py \
        -i '{npy_files[0]}' \
        --checkpoint-path '{VQGAN_CKPT}' \
        --output-path     '{NEW_OUTPUT}'
    if os.path.exists(NEW_OUTPUT):
        print('\n✅ Done!')
        ipd.display(ipd.Audio(NEW_OUTPUT))
    else:
        print('\n❌ Decoding failed.')

/content/fish-speech
🎙️  Generating: Сайн байна уу? Та хэрхэн байна?

2026-03-29 03:43:56.085 | INFO     | __main__:main:1056 - Loading model ...
Traceback (most recent call last):
  File "/content/fish-speech/fish_speech/models/text2semantic/inference.py", line 1117, in <module>
    main()
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1269, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 824, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/fish-speech/fish_speech/models/text2semantic/inf